# 11 · Variable sourcing and the empty-variable trap

## Goal

Wire the workflow's spend lookup to a real MCP tool result instead of a
mocked Dataverse query, understand the precedence rules across MCP results,
environment variables, and connection references, and add the guard that
stops the workflow from cheerfully continuing on an empty value.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from pathlib import Path
assert (Path("../agents/contract-renewal-desk/workflows/renewal-check.yaml")).exists(), "run 10 first"


## Concept

A workflow variable can come from three places: an MCP tool result, an
environment variable (per-environment config, resolved at deploy time), or
a connection reference (an authenticated binding to an external system).
Precedence matters when more than one *could* populate the same variable —
generally the most specific, most recently-resolved source wins, and this
notebook makes that concrete with the spend lookup.

**The failure this notebook exists to prevent:** a variable that resolves
to empty string, and a workflow that doesn't check — it just continues,
treats empty as "no flag," and silently recommends `renew` for a supplier
with no spend record on file at all, rather than surfacing a data problem.
`evals/golden_cases.json#wf-03-empty-variable-trap` targets exactly this.
The fix is never "hope the source populates it" — it's an explicit
not-empty check with a hard stop on failure.


## Build


In [ ]:
import yaml
from pathlib import Path
workflow_path = Path("../agents/contract-renewal-desk/workflows/renewal-check.yaml")
workflow = yaml.safe_load(workflow_path.read_text())

# Replace the mocked Dataverse lookup with a real MCP tool call (finance-ops-mcp, wired properly in 13)
for step in workflow["steps"]:
    if step["id"] == "lookupSpend":
        step["type"] = "mcp-tool-call"
        step["server"] = "finance-ops-mcp"
        step["tool"] = "getSupplierSpend"
        step["arguments"] = {"supplierName": "@{inputs.supplierName}"}

# The guard: hard stop on empty, inserted before the decide step
workflow["steps"].insert(
    [s["id"] for s in workflow["steps"]].index("decide"),
    {
        "id": "guardSpendPresent",
        "type": "if-else",
        "condition": "@{steps.lookupSpend.output.spendRecord} == null or @{steps.lookupSpend.output.spendRecord} == ''",
        "ifTrue": [{"id": "failFast", "type": "terminate", "reason": "spend record missing — cannot recommend renew/escalate without it"}],
        "ifFalse": [],
    },
)
workflow_path.write_text(yaml.dump(workflow, sort_keys=False))
print(workflow_path.read_text())


In [ ]:
from csx.pac import copilot_push
import subprocess
copilot_push(Path("../agents/contract-renewal-desk"))
subprocess.run(["pac", "copilot", "publish", "--name", "crd_contract-renewal-desk"], check=True)


### Precedence, made concrete

`supplierName` can arrive as: (1) an explicit workflow input from the calling turn, (2) an environment variable default for a single-supplier test environment, (3) never from a connection reference — connection references authenticate a call, they don't carry business data. Precedence here is simple because only inputs and env-var defaults compete: **explicit input always wins over an environment-variable default.**


## Verify

Same harness, same golden set, every notebook.


In [ ]:
from csx.clients import get_copilot_client
from csx.verify import run_suite, load_golden
from csx.cost import CreditMeter

client = get_copilot_client(settings, delegated=True)
meter = CreditMeter(environment_id=settings.get("DATAVERSE_ENV_ID"))

trap_case = [c for c in load_golden(tags=["empty-variable"])]
suite = run_suite(client, cases=trap_case, credit_meter=meter, min_pass_rate=1.0)

full_workflow_suite = run_suite(client, cases=load_golden(tags=["workflow"]), credit_meter=meter, min_pass_rate=0.75)


## Cost


In [ ]:
meter.report_cost("11", budget=settings.get("COPILOT_CREDIT_BUDGET"),
                   delta_credits=(suite.total_credits + full_workflow_suite.total_credits),
                   note="MCP-backed lookup + empty-variable guard verification")


## Teardown


In [ ]:
print("No teardown — the workflow's final shape from here persists through 25.")
